In [1]:
from pathlib import Path

#  --- data paths ---
GEOJSON = Path('../data/epc_abm_newcastle.geojson')
CLIMATE = Path('../data/ncc_2t_timeseries_2010_2039.parquet')
OUTDIR_QUICK = Path('results_quick')
OUTDIR_WIN   = Path('results_2020_2024')

for p in [GEOJSON, CLIMATE]:
    print(p, "exists:", p.exists())


../data/epc_abm_newcastle.geojson exists: True
../data/ncc_2t_timeseries_2010_2039.parquet exists: True


In [2]:
from household_energy.model import EnergyModel
from household_energy.climate import ClimateField
import pandas as pd
import geopandas as gpd
gdf = gpd.read_file(GEOJSON)

In [ ]:
from household_energy.model import EnergyModel
from household_energy.climate import ClimateField
import pandas as pd
import numpy as np

# paths & window
clim_path   = CLIMATE   # <-- your prepared hourly parquet
start_utc_i = pd.Timestamp("2020-01-01T00:00:00Z")
end_utc_e   = pd.Timestamp("2039-01-01T00:00:00Z")   # exclusive

# 1) Inspect climate & compute aligned hour range (like run.py)
cf = ClimateField(clim_path)
i0 = cf.time_index_for(start_utc_i)
i1 = cf.time_index_for(end_utc_e)
if i1 <= i0:
    raise ValueError("Window invalid for this climate file (end <= start).")
T_hours = int(i1 - i0)

# Align start to the climate grid's actual timestamp at i0 (exact hour on the parquet)
start_utc_aligned = pd.to_datetime(cf.times[i0], utc=True)   # aligns to grid:contentReference[oaicite:3]{index=3}
print(f"Aligned window: {start_utc_aligned} → {start_utc_aligned + pd.to_timedelta(T_hours, 'h')} (exclusive)")
print(f"Hours to simulate: {T_hours:,}")

# 2) Build the model (agent-level off for long runs; can turn on if you need it)
m = EnergyModel(
    gdf=gdf,
    climate_parquet=clim_path,
    climate_start=start_utc_aligned,
    local_tz="Europe/London",
    collect_agent_level=True,     # faster; set True if you need household traces
    agent_collect_every=24,
)

######
# edits to dial down energy spikes to calibration.
# — dial down occupant spikes from kWh/h to something realistic —
m.energy_per_person_home = 0.06   # ~1.44 kWh/day per person
m.energy_per_person_away = 0.01   # ~0.24 kWh/day per person

# — soften climate sensitivity —
m.heating_slope_kWh_per_deg = 0.03
m.cooling_slope_kWh_per_deg = 0.02

# — reduce the base to represent realistic away from home energy use. —
for h in m.household_agents:
    h.annual_energy_kwh *= 0.5    # adjust 0.4–0.6 until ABM≈DESNZ
    h.refresh_hourly_base()
######




Aligned window: 2020-01-01 00:00:00+00:00 → 2025-01-01 00:00:00+00:00 (exclusive)
Hours to simulate: 43,848


/Users/abeltran/Documents/GitHub/spdt_abm/household_energy/model.py:191: RuntimeWarning: Mean of empty slice
  np.nanmean([getattr(h, "ambient_tempC", np.nan) for h in m.household_agents])


In [ ]:
# --- 3) Timed run with progress updates --------------------------------------
import time, datetime

print_every = 24 * 7 * 52  # yearly progress (8760 h)

t0 = time.time()
print(f"Starting run for {T_hours:,} hours ...")

for h in range(T_hours):
    m.step()

    if (h + 1) % print_every == 0 or (h + 1) == T_hours:
        elapsed = time.time() - t0
        hrs_done = h + 1
        hrs_left = T_hours - hrs_done
        rate = elapsed / hrs_done
        eta = datetime.timedelta(seconds=hrs_left * rate)
        print(f"  progressed {hrs_done:,}/{T_hours:,} hours "
              f"({hrs_done/T_hours:5.1%}) "
              f"elapsed {datetime.timedelta(seconds=elapsed).total_seconds()/60:.1f} min "
              f"ETA {str(eta).split('.')[0]}")

t_total = time.time() - t0
print(f"✅ Windowed run complete in {datetime.timedelta(seconds=t_total)} "
      f"({t_total/60:.1f} min total)")
